In [0]:

# Step 1: Define the Bronze customers path

BRONZE = "abfss://lakehouse@saatliqcommerce01dl.dfs.core.windows.net/bronze"

# Step 2: Read Parquet files into a Spark DataFrame
df_customers = spark.read.parquet(f"{BRONZE}/customers")

# Step 3: Inspect the structure
df_customers.printSchema()

# Step 4: Display five records
display(df_customers.limit(5))


root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- updated_at: timestamp (nullable = true)



customer_id,customer_name,email,city,signup_date,updated_at
1,Nisha Gupta,nisha.gupta541@example.com,delhi,2024-01-02,2024-01-02T00:48:55.000Z
2,Tara Chopra,tara.chopra529@example.com,Bengaluru,2024-01-02,2024-01-02T10:44:55.000Z
3,Vivaan Chopra,vivaan.chopra414@example.com,Mumbai,2024-01-02,2024-01-02T21:00:42.000Z
4,Sai Rao,sai.rao859@example.com,Mumbai,2024-01-03,2024-01-03T06:44:44.000Z
5,Aman Sandhu,aman.sandhu622@example.com,Chennai,2024-01-03,2024-01-03T22:53:00.000Z


In [0]:

from pyspark.sql import functions as F

city_profile = (
    df_customers
    .groupBy("city")
    .count()
    .orderBy(F.desc("count"))
)

display(city_profile)


city,count
Mumbai,144
Bengaluru,143
Delhi,131
Hyderabad,100
Chennai,87
Pune,79
Kolkata,62
Ahmedabad,55
Jaipur,50
Surat,27


In [0]:

from pyspark.sql import functions as F

customers_clean = (
    df_customers
    .withColumn(
        "city",
        F.initcap(F.trim(F.col("city")))
    )
)

display(
    customers_clean
    .groupBy("city")
    .count()
    .orderBy("city")
)


city,count
Ahmedabad,61
Bangalore,13
Bengaluru,166
Bombay,12
Calcutta,1
Chennai,91
Delhi,146
Hyderabad,109
Jaipur,57
Kolkata,71


In [0]:

customers_clean = (
    customers_clean
    .withColumn(
        "city",
        F.when(F.col("city") == "Bangalore", "Bengaluru")
         .when(F.col("city") == "Bombay", "Mumbai")
         .when(F.col("city") == "Madras", "Chennai")
         .when(F.col("city") == "Calcutta", "Kolkata")
         .otherwise(F.col("city"))
    )
)

display(
    customers_clean
    .groupBy("city")
    .count()
    .orderBy("city")
)


city,count
Ahmedabad,61
Bengaluru,179
Chennai,94
Delhi,146
Hyderabad,109
Jaipur,57
Kolkata,72
Mumbai,169
Pune,89
Surat,32


In [0]:

customers_clean = (
    customers_clean
    .withColumn(
        "email",
        F.lower(F.trim(F.col("email")))
    )
)

display(customers_clean.select("customer_id", "email").limit(10))


customer_id,email
1,nisha.gupta541@example.com
2,tara.chopra529@example.com
3,vivaan.chopra414@example.com
4,sai.rao859@example.com
5,aman.sandhu622@example.com
6,diya.banerjee184@example.com
7,siddharth.reddy445@example.com
8,joseph.sharma744@example.com
9,mohammed.kapoor173@example.com
10,rahul.singh986@example.com


In [0]:

customers_clean = (
    customers_clean
    .withColumn(
        "is_test",
        F.when(
            F.col("email").endswith("@atliq-test.com"),
            True
        ).otherwise(False)
    )
)

# Check number of test and real customers
display(
    customers_clean
    .groupBy("is_test")
    .count()
)


is_test,count
true,8
false,1000


In [0]:

# Final customer ID validation and date conversion

customers_clean = (
    customers_clean
    .withColumn("signup_date", F.to_date(F.col("signup_date")))
    .filter(F.col("customer_id").isNotNull())
    .dropDuplicates(["customer_id"])
)

# Verify final structure
customers_clean.printSchema()

# Verify customer counts
print("Total customers:", customers_clean.count())

print(
    "Unique customer IDs:",
    customers_clean.select("customer_id").distinct().count()
)

print(
    "Distinct cities:",
    customers_clean.select("city").distinct().count()
)

display(customers_clean.limit(10))


root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- is_test: boolean (nullable = false)

Total customers: 1008
Unique customer IDs: 1008
Distinct cities: 10


customer_id,customer_name,email,city,signup_date,updated_at,is_test
12,Isha Pillai,isha.pillai165@example.com,Hyderabad,2024-01-09,2024-01-09T06:52:09.000Z,false
18,Anjali Banerjee,anjali.banerjee743@example.com,Delhi,2024-01-13,2024-01-13T00:48:31.000Z,false
38,Varun Chopra,varun.chopra698@example.com,Hyderabad,2024-02-01,2024-02-01T16:30:27.000Z,false
67,Kavya Verma,kavya.verma720@example.com,Kolkata,2024-03-18,2024-03-18T12:02:13.000Z,false
70,Riya Pillai,riya.pillai304@example.com,Jaipur,2024-03-23,2024-03-23T12:08:35.000Z,false
93,Karan Pillai,karan.pillai219@example.com,Ahmedabad,2024-04-09,2024-04-09T17:14:37.000Z,false
161,Nikhil Gill,nikhil.gill911@example.com,Pune,2024-06-13,2024-06-13T20:33:48.000Z,false
186,Aishwarya Patel,aishwarya.patel670@example.com,Pune,2024-07-16,2024-07-16T05:52:44.000Z,false
190,Rohan Agarwal,rohan.agarwal554@example.com,Bengaluru,2024-07-17,2024-07-17T07:58:20.000Z,false
218,Reyansh Mehta,reyansh.mehta939@example.com,Delhi,2024-08-16,2024-08-16T07:28:22.000Z,false


In [0]:

# Write cleaned customers into our Silver Delta table

customers_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("atliq_dl.silver.customers")

print("Silver customers table created successfully!")


Silver customers table created successfully!


In [0]:
%run ./utils_dq

In [0]:
raw = spark.read.parquet(f"{BRONZE}/customers")
silver = spark.table("atliq_dl.silver.customers")

r = raw.dropDuplicates(["customer_id"])
j = r.alias("r").join(silver.alias("s"), "customer_id")

log_dq("customers", "city_standardized",
       j.filter(~F.col("r.city").eqNullSafe(F.col("s.city"))).count())

log_dq("customers", "email_normalized",
       j.filter(~F.col("r.email").eqNullSafe(F.col("s.email"))).count())

log_dq("customers", "test_accounts_flagged",
       silver.filter(F.col("is_test") == True).count())

log_dq("customers", "duplicate_customer_ids_removed",
       raw.count() - r.count())

display(
    spark.table("atliq_dl.silver.dq_log")
    .filter("table_name = 'customers'")
    .orderBy("rule")
)

run_date,table_name,rule,rows_affected
2026-10-31,customers,city_standardized,130
2026-11-01,customers,city_standardized,130
2026-10-31,customers,duplicate_customer_ids_removed,0
2026-11-01,customers,duplicate_customer_ids_removed,0
2026-10-31,customers,email_normalized,52
2026-11-01,customers,email_normalized,52
2026-10-31,customers,test_accounts_flagged,8
2026-11-01,customers,test_accounts_flagged,8
